# The Threshold/Stump: Sliding the Cutoff

**Week 2 · Lesson 5** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week2/lectures/05_threshold_stump.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Adapted from ISLP Ch. 4.4 ("Varying the threshold") where relevant.*

## Setup

Same boilerplate as every notebook this term — see Lesson 8 if this is your
first time seeing it.

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

## 1. Business Problem

**Why this matters:** every metric in Lesson 4 — precision, recall, ROC,
PR — reduces to the same object underneath: one continuous score, and a
line drawn somewhere on it. In production that line is a real decision: a
fraud score above 0.7 blocks a transaction, a churn score above 0.4 triggers
a retention call. Nudge the line half a point and every downstream count —
false alarms, missed cases — changes with it. This notebook builds Lesson
4's ROC and PR curves by hand, sliding one threshold across one score, so
"score → decision" stops being a slide and becomes something you can
watch happen.

## 2. Data

A single continuous **score** per case (think: predicted churn probability,
fraud risk, credit risk) plus the true class it belongs to. Generated by
`make_classification_sandbox` from the course's synthetic-data sandbox
(`gbus564_coursework.utils.sandbox`, toured in Week 1 Lesson 9) — `noise` controls how much
the two classes' scores overlap, and `imbalance` sets the positive rate.

In [ ]:
from gbus564_coursework.utils.sandbox import make_classification_sandbox

X, y = make_classification_sandbox(n_samples=1000, noise=1.2, imbalance=0.2, rng=rng)
df = X.assign(y=y)
print(f"positive rate: {y.mean():.1%}")
df.head()

In [ ]:
fig, ax = plt.subplots()
ax.hist(df.loc[df["y"] == 0, "score"], bins=30, alpha=0.6, label="class 0 (negative)")
ax.hist(df.loc[df["y"] == 1, "score"], bins=30, alpha=0.6, label="class 1 (positive)")
ax.set_xlabel("score")
ax.set_ylabel("count")
ax.set_title("The two classes overlap — no threshold separates them perfectly")
ax.legend()
plt.tight_layout()

## 3. Model

There's no algorithm to fit here — the "model" *is* the threshold rule
itself, sometimes called a **decision stump**: predict positive if the
score clears a cutoff, negative otherwise. The baseline every threshold
choice implicitly competes with is the trivial ones — a cutoff below every
score (flags everyone) or above every score (flags no one).

In [ ]:
def stump_predict(score, threshold):
    return (score >= threshold).astype(int)

# Three cutoffs, low to high, to see the same rule swing from "flag almost
# everyone" to "flag almost no one."
thresholds_to_inspect = [0.5, 1.0, 1.5]
for t in thresholds_to_inspect:
    flagged_rate = stump_predict(df["score"], t).mean()
    print(f"threshold={t:.1f}  ->  {flagged_rate:.1%} of cases flagged positive")

## 4. Evaluate

*Score against the baseline, not in isolation — a metric only means
something next to what it's beating.*

First, the confusion matrix at each of the three cutoffs above — the exact
same four buckets from Lesson 4, recomputed as the line moves. Then, every
threshold at once: sweeping the cutoff from the lowest score to the highest
traces out the ROC and PR curves from Lesson 4's slides.

In [ ]:
from sklearn.metrics import confusion_matrix

EPS = 1e-9  # tiny epsilon so a threshold that flags nobody doesn't divide by zero

for t in thresholds_to_inspect:
    preds = stump_predict(df["score"], t)
    tn, fp, fn, tp = confusion_matrix(df["y"], preds).ravel()
    print(f"threshold={t:.1f}:  TP={tp:3d}  FP={fp:3d}  FN={fn:3d}  TN={tn:3d}"
          f"   precision={tp / (tp + fp + EPS):.2f}  recall={tp / (tp + fn + EPS):.2f}")

**What the three cutoffs just showed.** Raising the threshold from 0.5 to
1.5 cuts the share of cases flagged from 46% to 22%. That one move removes
192 false alarms (FP 281 → 89) but also misses 48 more real positives
(TP 175 → 127). Precision rises from 0.38 to 0.59 while recall falls from
0.87 to 0.63. Neither cutoff is "better" — each buys fewer false alarms
with more missed cases, and which side of that trade you want depends on
what each mistake costs.

**Against the baseline.** Only 20% of cases are positive. Flagging everyone
gets perfect recall at a precision of 0.20, the base rate. So t=0.5 is
almost twice as precise as flagging blindly, and t=1.5 almost three times.
That ratio, not 0.38 or 0.59 on its own, tells you the score is doing
something useful.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, average_precision_score

fpr, tpr, roc_thresh = roc_curve(df["y"], df["score"])
auc = roc_auc_score(df["y"], df["score"])
prec, rec, pr_thresh = precision_recall_curve(df["y"], df["score"])
ap = average_precision_score(df["y"], df["score"])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(fpr, tpr, lw=2.5, label=f"AUC={auc:.2f}")
axes[0].plot([0, 1], [0, 1], "--", color="gray", label="random guessing")
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve"); axes[0].legend()

axes[1].plot(rec, prec, lw=2.5, label=f"AP={ap:.2f}")
axes[1].axhline(df["y"].mean(), color="gray", linestyle="--", label="random guessing")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve"); axes[1].legend()

# Mark the three inspected thresholds on both curves so a slide of the
# cutoff visibly traces a point moving along each curve.
for t in thresholds_to_inspect:
    preds = stump_predict(df["score"], t)
    tn, fp, fn, tp = confusion_matrix(df["y"], preds).ravel()
    t_fpr, t_tpr = fp / (fp + tn), tp / (tp + fn)
    t_prec = tp / (tp + fp + EPS)
    axes[0].scatter([t_fpr], [t_tpr], color="black", zorder=5)
    axes[0].annotate(f"t={t}", (t_fpr, t_tpr), textcoords="offset points", xytext=(6, -3))
    axes[1].scatter([t_tpr], [t_prec], color="black", zorder=5)
    axes[1].annotate(f"t={t}", (t_tpr, t_prec), textcoords="offset points", xytext=(6, -3))

plt.tight_layout()

**From three points to every point.** Each cutoff is one point on each
curve above: (false positive rate, true positive rate) on the ROC curve
and (recall, precision) on the PR curve. At t=0.5 the ROC point is about
(0.35, 0.87); at t=1.5 it moves to about (0.11, 0.63). Sweep the threshold
across every score instead of just three and those points join into the
full curves. The black dots mark the three cutoffs from above, so you can
see where they land.

**What to look for in the plots:**

- **ROC** — a higher threshold moves you *down and left* (fewer false
  alarms, fewer catches). The dashed diagonal is a score with no
  information. **AUC** (≈0.87 here) is the chance that a randomly chosen
  positive scores higher than a randomly chosen negative.
- **PR** — a higher threshold moves you *left and up* (fewer catches, but
  the ones you flag are more often right). The dashed baseline sits at the
  20% positive rate, not at 0.5, which makes PR the more honest view when
  positives are rare. **AP** (≈0.68 here) summarizes the whole curve.
- **One caveat** — AUC and AP grade the score across *every* threshold at
  once, but in production you deploy only one. They tell you whether the
  score is worth using; they don't tell you where to put the line.

## 5. Interpret

*What is the model actually keying off of? Business-readable, not just
statistically significant.*

The histogram from Section 2 already contains every confusion-matrix count
— a threshold line just carves it into four regions. Moving the line to the
right shrinks false positives but grows false negatives; moving it left
does the reverse. That's the entire mechanism behind every point on the
ROC/PR curves above.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, t in zip(axes, thresholds_to_inspect):
    for cls, color in [(0, "#B33A3A"), (1, "#1A73E8")]:
        vals = df.loc[df["y"] == cls, "score"]
        ax.hist(vals, bins=30, alpha=0.6, color=color,
                label="class 0 (negative)" if cls == 0 else "class 1 (positive)")
    ax.axvline(t, color="black", linewidth=2)
    ax.set_title(f"threshold = {t}")
    ax.set_xlabel("score")
axes[0].set_ylabel("count")
axes[0].legend(fontsize=8)
plt.suptitle("Everything right of the line is predicted positive")
plt.tight_layout()

## 6. Recommend

**The rule:** there is no single "best" threshold — only the threshold that
matches the relative cost of a false positive against a false negative for
*this* decision. Sliding it left trades precision for recall; sliding it
right does the reverse; the ROC and PR curves are the full menu of that
tradeoff, not a search for one right answer on it. Before locking in a
cutoff for any classifier this term, sweep it across the full range first —
exactly as this notebook did — rather than defaulting to 0.5.

**Caveats / next steps:** this sandbox's score is drawn from two overlapping
Gaussians by construction — real scores can be far messier (multimodal,
miscalibrated). The *mechanism* generalizes regardless: Week 3 Lesson 4
("Score → Decision") replays this exact stump with a real cost attached to
each quadrant of the confusion matrix, so the "right" threshold stops being
a menu and becomes a specific number.